# Handling Class Imbalance: Downsampling, SMOTE, and Beyond

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/class_imbalance.ipynb)

Compare five strategies for imbalanced classification and evaluate with precision-recall curves.

**Blog post:** [Handling Class Imbalance](https://sesen.ai/blog/class-imbalance-downsampling-smote-comprehensive-guide)

**Key papers:**
- Prentice, R.L. & Pyke, R. (1979). Logistic Disease Incidence Models and Case-Control Studies. *Biometrika*, 66(3), 403-411.
- Chawla, N.V. et al. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *JAIR*, 16, 321-357.

In [ ]:
!pip install -q imbalanced-learn

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             precision_recall_curve, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from collections import Counter

np.random.seed(42)

## 1. Create an Imbalanced Dataset

95% class 0, 5% class 1 — typical for fraud detection, rare disease screening, etc.

In [ ]:
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10, n_redundant=5,
    n_classes=2, weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Class distribution: {Counter(y)}")
print(f"Train: {Counter(y_train)}, Test: {Counter(y_test)}")

## 2. Prepare Resampled Datasets

In [ ]:
majority_idx = np.where(y_train == 0)[0]
minority_idx = np.where(y_train == 1)[0]

# Random downsampling
down_idx = np.random.choice(majority_idx, size=len(minority_idx), replace=False)
X_down = np.vstack([X_train[down_idx], X_train[minority_idx]])
y_down = np.concatenate([y_train[down_idx], y_train[minority_idx]])

# Random upsampling
up_idx = np.random.choice(minority_idx, size=len(majority_idx), replace=True)
X_up = np.vstack([X_train[majority_idx], X_train[up_idx]])
y_up = np.concatenate([y_train[majority_idx], y_train[up_idx]])

# SMOTE
X_smote, y_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)

print(f"Original:    {Counter(y_train)}")
print(f"Downsampled: {Counter(y_down)}")
print(f"Upsampled:   {Counter(y_up)}")
print(f"SMOTE:       {Counter(y_smote)}")

## 3. Train and Evaluate All Strategies

In [ ]:
strategies = {
    'Baseline':      (X_train, y_train, {}),
    'Downsample':    (X_down, y_down, {}),
    'Upsample':      (X_up, y_up, {}),
    'SMOTE':         (X_smote, y_smote, {}),
    'Class Weights': (X_train, y_train, {'class_weight': 'balanced'}),
}

models = {}
print(f"{'Method':<15} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6}")
print("-" * 45)
for name, (X_tr, y_tr, kwargs) in strategies.items():
    clf = LogisticRegression(max_iter=1000, random_state=42, **kwargs)
    clf.fit(X_tr, y_tr)
    models[name] = clf
    pred = clf.predict(X_test)
    print(f"{name:<15} {precision_score(y_test, pred):>6.3f} "
          f"{recall_score(y_test, pred):>6.3f} "
          f"{f1_score(y_test, pred):>6.3f} "
          f"{np.mean(pred == y_test):>6.3f}")

## 4. Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#9E9E9E', '#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for (name, clf), color in zip(models.items(), colors):
    y_prob = clf.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')

ax.axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, (name, clf) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, clf.predict(X_test))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Neg', 'Pos']); ax.set_yticklabels(['Neg', 'Pos'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=12,
                    fontweight='bold', color='white' if cm[i, j] > cm.max()/2 else 'black')
axes[0].set_ylabel('True Label')
fig.suptitle('Confusion Matrices', fontsize=14, y=1.05)
fig.tight_layout()
plt.show()

## Exercises

1. **Tune the threshold.** Train the baseline model and sweep the decision threshold from 0.1 to 0.9. Plot precision and recall vs threshold. Can you find a threshold giving better F1 than SMOTE?

2. **Try Borderline-SMOTE.** Replace `SMOTE()` with `BorderlineSMOTE()` from imbalanced-learn. Does it improve F1?

3. **Use a tree-based model.** Replace LogisticRegression with RandomForestClassifier. Do resampling strategies still help?

4. **Increase imbalance.** Change `weights=[0.99, 0.01]` for a 99:1 ratio. How do the strategies compare with extreme imbalance?